In [11]:
import sys
sys.path.append('/content/drive/MyDrive/pythonprojects/snake-game/pygame_snake_game_gg_colab')

In [13]:

import numpy as np
import random
import pygame
import sys
import os
import math
import torch
import torch.nn as nn
import torch.optim as optim
from snake_game import SnakeGame
from collections import deque

import csv

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
# Game Constants
WIDTH, HEIGHT = 800, 600
BLOCK_SIZE = 20
GRID_WIDTH = WIDTH // BLOCK_SIZE
GRID_HEIGHT = HEIGHT // BLOCK_SIZE

# Neural Network Parameters
STATE_SIZE = 4  # grid_x, grid_y, food_dir, danger_level
BATCH_SIZE = 128
MEMORY_SIZE = 10000
GAMMA = 0.95
EPSILON_START = 1.0
EPSILON_END = 0.01
EPSILON_DECAY = 0.995
LEARNING_RATE = 0.001

In [16]:
# Initialize Game
game = SnakeGame(width=WIDTH, height=HEIGHT)
screen = pygame.Surface((WIDTH, HEIGHT))  # Off-screen rendering
clock = pygame.time.Clock()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


In [17]:
def save_weights_to_csv(state_dict, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, mode='w', newline='') as file:
        writer = csv.writer(file)
        for key, weight in state_dict.items():
            writer.writerow([key])
            flat_weights = weight.flatten().tolist()
            writer.writerow(flat_weights)
            writer.writerow([])
    print(f"Weights saved to {path}")

In [18]:
class DQN(nn.Module):
    """Deep Q-Network with state representation"""

    def __init__(self, input_size, output_size):
        super(DQN, self).__init__()
        # Feed-forward
        self.fc = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, output_size)
        )

    def forward(self, x):
        return self.fc(x)


In [19]:
class DQNAgent:
    def __init__(self):
        # Policy Network
        self.policy_net = DQN(STATE_SIZE, 4)  # Policy_Network
        self.target_net = DQN(STATE_SIZE, 4)  # Target_network
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=LEARNING_RATE)
        self.memory = deque(maxlen=MEMORY_SIZE)
        self.epsilon = EPSILON_START
        self.steps_done = 0

        # Initialize target network
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()

    def get_state(self, game_state):
        """state representation with full danger detection"""
        head = game_state['snake_head']
        food = game_state['food']
        body = game_state['snake_body']

        # Normalized grid position
        grid_x = head[0] // BLOCK_SIZE
        grid_y = head[1] // BLOCK_SIZE

        # Food direction (0-3: up, right, down, left)
        dx, dy = food[0] - head[0], food[1] - head[1]
        food_dir = (1 if dx > 0 else 3) if abs(dx) > abs(dy) else (2 if dy > 0 else 0)

        # Danger detection in all 4 directions
        danger = 0
        # Check left, right, up, down
        for x, y in [
            (head[0] - BLOCK_SIZE, head[1]),  # Left
            (head[0] + BLOCK_SIZE, head[1]),  # Right
            (head[0], head[1] - BLOCK_SIZE),  # Up
            (head[0], head[1] + BLOCK_SIZE)  # Down
        ]:
            if (x < 0 or x >= WIDTH or y < 0 or y >= HEIGHT) or ((x, y) in body):
                danger += 1

        # Normalized danger level (0-1.0)
        danger_level = danger / 4

        return torch.FloatTensor([
            grid_x / GRID_WIDTH,
            grid_y / GRID_HEIGHT,
            food_dir / 3,
            danger_level
        ])

    def remember(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def act(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, 3)

        with torch.no_grad():
            q_values = self.policy_net(state)
            return q_values.argmax().item()

    def learn(self):
        if len(self.memory) < BATCH_SIZE:
            return

        batch = random.sample(self.memory, BATCH_SIZE)
        # unpacks them into separate lists
        states, actions, rewards, next_states, dones = zip(*batch)

        # Convert to tensors and move to GPU
        states = torch.stack(states).to(device)
        actions = torch.LongTensor(actions).to(device)
        rewards = torch.FloatTensor(rewards).to(device)
        next_states = torch.stack(next_states).to(device)
        dones = torch.FloatTensor(dones).to(device)

        # Current Q values
        current_q = self.policy_net(states).gather(1, actions.unsqueeze(1))

        # Target Q values
        with torch.no_grad():
            # next_q = self.target_net(next_states).max(1)[0]
            # Double DQN
            next_actions = self.policy_net(next_states).argmax(1)
            next_q = self.target_net(next_states).gather(1, next_actions.unsqueeze(1)).squeeze()
            target_q = rewards + (1 - dones) * GAMMA * next_q

        # Compute loss
        loss = nn.MSELoss()(current_q.squeeze(), target_q)

        # Optimize the model
        self.optimizer.zero_grad()
        # Back propagation
        loss.backward()
        self.optimizer.step()

        # Decay epsilon
        self.epsilon = max(EPSILON_END, self.epsilon * EPSILON_DECAY)

        # Update target network periodically
        if self.steps_done % 100 == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())

        self.steps_done += 1

In [20]:
agent = DQNAgent()
agent.policy_net.to(device)
agent.target_net.to(device)
scores = []
mean_scores = []
best_mean_score = float('-inf')

# TODO: Change the file name
# Model Loading
WEIGHT_PATH = '/content/drive/MyDrive/pythonprojects/snake-game/pygame_snake_game_gg_colab/Current DQN WEIGHTS/snake_dqn.pth'
RETRAIN = True
if os.path.exists(WEIGHT_PATH):
    # soon In pytouch, this code below would not be able to run without this {weights_only = True}, check for the
    # updates overtime.
    # agent.policy_net.load_state_dict(torch.load(WEIGHT_PATH), weights_only = True)
    agent.epsilon = 0.2 if RETRAIN else EPSILON_END
    agent.policy_net.load_state_dict(torch.load(WEIGHT_PATH, map_location=device))
    agent.target_net.load_state_dict(agent.policy_net.state_dict())
    print("Loaded saved weights")

# Training Loop
for episode in range(5000):
    current_score = game.get_state()["score"]
    high_score = game.get_state()["highscore"]
    state = game.reset()
    current_state = agent.get_state(state).to(device)
    total_reward = 0
    done = False


    while not done:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                # torch.save(agent.policy_net.state_dict(), WEIGHT_PATH)
                # print("Saved weights in .pth")
                pygame.quit()
                sys.exit()

        action = agent.act(current_state)
        next_state, reward, done = game.step(action)
        # reward = reward.to(device)
        next_state_processed = agent.get_state(next_state).to(device) # get the processed state and move to GPU

        # Store experience with negative reward for collisions
        agent.remember(current_state, action, reward, next_state_processed, done)
        agent.learn()

        current_state = next_state_processed
        total_reward += reward

        # Rendering
        game.render(screen, clock.get_fps())
        pygame.display.flip()
        clock.tick(240)  # Reduce speed for better observation

    # Episode statistics
    scores.append(total_reward)
    mean_score = np.mean(scores[-100:])
    mean_scores.append(mean_score)

    # Save best model
    if mean_score > best_mean_score:
        best_mean_score = mean_score
        torch.save(agent.policy_net.state_dict(), WEIGHT_PATH)
        print("Saved new weights")

    # Save weights to CSV every 500 episodes
    # if episode % 500 == 0 and episode != 0:
    #     save_weights_to_csv(agent.policy_net.state_dict(), "csv files/DQN-Weights.csv")

    print(f"Ep {episode:04d} | Score: {total_reward:3.0f} | ε: {agent.epsilon:.3f} | Mean: {mean_score:.1f} | "
          f"highscore : {high_score} | current_score: {current_score}")


# Final Save at the 5000 episode
# torch.save(agent.policy_net.state_dict(), WEIGHT_PATH)

# Note Csv files of the weights are large
# Save to CSV
# save_weights_to_csv(agent.policy_net.state_dict(), "csv files/DQN-Weights.csv")
# print("Saved weights to scv file")
pygame.quit()

Streaming output truncated to the last 5000 lines.
Ep 0060 | Score: -65 | ε: 0.010 | Mean: -73.5 | highscore : 5 | current_score: 0
Ep 0061 | Score: -72 | ε: 0.010 | Mean: -73.5 | highscore : 5 | current_score: 0
Ep 0062 | Score: -59 | ε: 0.010 | Mean: -73.3 | highscore : 5 | current_score: 0
Ep 0063 | Score: -124 | ε: 0.010 | Mean: -74.1 | highscore : 5 | current_score: 0
Ep 0064 | Score: -67 | ε: 0.010 | Mean: -74.0 | highscore : 5 | current_score: 0
Ep 0065 | Score: -35 | ε: 0.010 | Mean: -73.4 | highscore : 5 | current_score: 1
Ep 0066 | Score: -191 | ε: 0.010 | Mean: -75.1 | highscore : 5 | current_score: 0
Ep 0067 | Score:  88 | ε: 0.010 | Mean: -72.7 | highscore : 5 | current_score: 0
Ep 0068 | Score: -67 | ε: 0.010 | Mean: -72.7 | highscore : 5 | current_score: 4
Ep 0069 | Score:  35 | ε: 0.010 | Mean: -71.1 | highscore : 5 | current_score: 0
Ep 0070 | Score: -87 | ε: 0.010 | Mean: -71.3 | highscore : 5 | current_score: 1
Ep 0071 | Score:  60 | ε: 0.010 | Mean: -69.5 | highscor